# Extension 4: Cohort Retention Forecasting

**Goal:** Given the first two weeks of a new cohort's retention data, predict their
week-8 and week-12 retention before those dates arrive.

**Models compared:**
- `LinearRegression` — baseline; fast and interpretable
- `GBTRegressor` — gradient-boosted trees; expected to capture non-linear decay patterns

**Data source:** `cohort_retention` (cohort-level weekly retention rates)

**Feature engineering:** Each training sample is one cohort. Features are early-week
retention rates (weeks 0–4); the target is a later-week retention rate (week 8 or 12).

---
**Prerequisites:** Run `make run-jobs` before opening this notebook.  
The `cohort_retention` table must have ≥ 1 cohort with data through week 12.

## Cell 1 — Setup

In [ ]:
import os
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    with psycopg2.connect(**PG) as conn:
        return pd.read_sql(sql, conn)

spark = (
    SparkSession.builder
    .appName("ML Feasibility — Retention Forecasting")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Load Cohort Retention Data

In [ ]:
cohort_pd = pg_query("""
    SELECT cohort_week, week_number, cohort_size, retention_rate
    FROM cohort_retention
    ORDER BY cohort_week, week_number
""")

print(f"Rows loaded       : {len(cohort_pd):,}")
print(f"Cohorts           : {cohort_pd['cohort_week'].nunique()}")
print(f"Max week observed : {cohort_pd['week_number'].max()}")
cohort_pd.head(8)

## Cell 3 — Feature Engineering

Pivot the long-format cohort table into wide format so each row is one cohort and
each column is a weekly retention rate. Early weeks become features; a later week
becomes the regression target.

In [ ]:
pivot = (
    cohort_pd
    .pivot_table(index="cohort_week", columns="week_number", values="retention_rate")
    .reset_index()
)
pivot.columns = [f"week_{c}" if isinstance(c, int) else c for c in pivot.columns]

# Attach cohort sizes (week_number == 0 row holds the original cohort size)
sizes = cohort_pd[cohort_pd["week_number"] == 0][["cohort_week", "cohort_size"]]
pivot = pivot.merge(sizes, on="cohort_week")

print(f"Pivoted shape: {pivot.shape}")
print("Columns:", list(pivot.columns))
pivot.head()

## Cell 4 — Check Data Availability

We need at least some cohorts with data through week 12. If week 12 is not available,
fall back to forecasting week 8.

In [ ]:
available_weeks = [c for c in pivot.columns if c.startswith("week_") and c != "week_"]
print("Available week columns:", sorted(available_weeks))

TARGET = "week_12" if "week_12" in pivot.columns else "week_8"
FEATURE_COLS = [c for c in ["week_0", "week_1", "week_2", "week_4", "cohort_size"] if c in pivot.columns]

print(f"\nTarget column : {TARGET}")
print(f"Feature cols  : {FEATURE_COLS}")

# Drop cohorts where any feature or target is missing
train_pivot = pivot.dropna(subset=FEATURE_COLS + [TARGET])
print(f"\nCohorts usable for training: {len(train_pivot)} of {len(pivot)}")

if len(train_pivot) < 4:
    print("WARNING: very few training samples. Generate more data with 'make generate-data'.")

## Cell 5 — MLlib Pipeline: LinearRegression vs GBTRegressor

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression

sdf = spark.createDataFrame(
    train_pivot[FEATURE_COLS + [TARGET, "cohort_week"]]
    .rename(columns={TARGET: "label"})
)

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")
evaluator = RegressionEvaluator(labelCol="label", metricName="rmse")
r2_eval   = RegressionEvaluator(labelCol="label", metricName="r2")

train, test = sdf.randomSplit([0.8, 0.2], seed=42)
print(f"Train cohorts: {train.count()}   Test cohorts: {test.count()}")
print()

best_model = None
best_rmse  = float("inf")
best_name  = ""

regressors = [
    ("LinearRegression", LinearRegression(featuresCol="features", labelCol="label")),
    ("GBTRegressor",     GBTRegressor(
        featuresCol="features", labelCol="label",
        maxIter=50, maxDepth=4, seed=42,
    )),
]

for name, regressor in regressors:
    pipe  = Pipeline(stages=[assembler, regressor])
    m     = pipe.fit(train)
    preds = m.transform(test)
    rmse  = evaluator.evaluate(preds)
    r2    = r2_eval.evaluate(preds)
    print(f"{name:22s}  RMSE={rmse:.4f}  R²={r2:.4f}")
    if rmse < best_rmse:
        best_rmse  = rmse
        best_model = m
        best_name  = name

print(f"\nBest model: {best_name}  (RMSE={best_rmse:.4f})")

## Cell 6 — Forecast Cohorts with Only Early-Week Data Available

This simulates the production use-case: new cohorts have data through week 4 only;
we predict their week-12 retention before it can be observed.

In [ ]:
new_cohorts_pd = pg_query("""
    SELECT cohort_week, week_number, cohort_size, retention_rate
    FROM cohort_retention
    WHERE week_number <= 4
""")

new_pivot = (
    new_cohorts_pd
    .pivot_table(index="cohort_week", columns="week_number", values="retention_rate")
    .reset_index()
)
new_pivot.columns = [f"week_{c}" if isinstance(c, int) else c for c in new_pivot.columns]
new_pivot = new_pivot.merge(
    new_cohorts_pd[new_cohorts_pd["week_number"] == 0][["cohort_week", "cohort_size"]],
    on="cohort_week",
)

# Keep only rows where all feature columns are available
new_pivot = new_pivot.dropna(subset=FEATURE_COLS)
print(f"Cohorts to forecast: {len(new_pivot)}")

if len(new_pivot) > 0:
    forecast_sdf = spark.createDataFrame(new_pivot[FEATURE_COLS + ["cohort_week"]])
    forecasts = best_model.transform(forecast_sdf)
    result = forecasts.select("cohort_week", "prediction").toPandas()
    result.columns = ["cohort_week", f"predicted_{TARGET}"]
    result[f"predicted_{TARGET}"] = result[f"predicted_{TARGET}"].round(4)
    print(f"\nForecasted {TARGET} retention rates:")
    print(result.to_string(index=False))
else:
    print("No cohorts with complete early-week data to forecast.")

## Cell 7 — Retention Curve Visualisation

Plot the observed retention curves for each cohort to visually confirm the forecasting context.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

for cohort, group in cohort_pd.groupby("cohort_week"):
    ax.plot(
        group["week_number"],
        group["retention_rate"],
        alpha=0.5,
        linewidth=1,
        label=str(cohort),
    )

ax.set_xlabel("Weeks since cohort start")
ax.set_ylabel("Retention rate")
ax.set_title("Cohort Retention Curves")

# Only show legend if manageable number of cohorts
if cohort_pd["cohort_week"].nunique() <= 15:
    ax.legend(title="Cohort", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)

plt.tight_layout()
plt.show()

## Cell 8 — Predicted vs Actual (Test Set)

In [ ]:
if test.count() > 0:
    test_preds = best_model.transform(test).select("label", "prediction", "cohort_week").toPandas()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(test_preds["label"], test_preds["prediction"], alpha=0.7)

    lo = min(test_preds[["label", "prediction"]].min())
    hi = max(test_preds[["label", "prediction"]].max())
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="perfect forecast")

    for _, row in test_preds.iterrows():
        ax.annotate(str(row["cohort_week"]), (row["label"], row["prediction"]), fontsize=7)

    ax.set_xlabel(f"Actual {TARGET} retention")
    ax.set_ylabel(f"Predicted {TARGET} retention")
    ax.set_title(f"Predicted vs Actual — {best_name} (RMSE={best_rmse:.4f})")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Test set is empty — skipping scatter plot.")

## Cell 9 — Cleanup

In [ ]:
spark.stop()
print("Spark session stopped.")